In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [2]:
def calculate_d1_d2(S, K, T, r, sigma):

    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    return d1,d2

In [3]:
def black_scholes_price(S, K, T, r, sigma):  

    d1, d2 = calculate_d1_d2(S, K, T, r, sigma)

    C = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    P = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return C,P

In [ ]:
def black_scholes_greeks(S, K, T, r, sigma):

    d1,d2 = calculate_d1_d2(S, K, T, r, sigma)

    delta_c = norm.cdf(d1)
    delta_p = -(norm.cdf(-d1))

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    vega = S * norm.pdf(d1) * np.sqrt(T)

    theta_c = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r*T) * norm.cdf(d2)
    theta_p = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) + r * K * np.exp(-r*T) * norm.cdf(-d2)

    call_greeks = {
        "delta": delta_c,
        "gamma": gamma,
        "vega": vega,
        "theta": theta_c
        }

    put_greeks = {
        "delta": delta_p,
        "gamma": gamma,
        "vega": vega,
        "theta": theta_p
    }

    return call_greeks, put_greeks

In [24]:
S0 = 100
sigma = 0.20
r = 0.05
T = 1
K = 100
n = 100
M = 100000
t_array = np.linspace(0, T, n+1)
rng = np.random.default_rng(4)

delta_t = T/n

In [25]:
# generates 10 (N) random paths

randoms = rng.normal(0, np.sqrt(delta_t), size = (M, n))
zeros = np.array([[0]] * M)

brownian_increments = np.concatenate((zeros, randoms), axis = 1)
brownian = np.cumsum(brownian_increments, axis = 1)

diffusion = np.multiply(sigma, brownian)
drift = np.multiply(t_array, (r - 0.5 * (sigma**2)))

price_factor = np.exp(drift + diffusion)
prices = np.multiply(S0, price_factor)


In [ ]:
# consider path1

path1 = prices[0]
delta = []

for i in range(len(path1) - 1):

    tau = T - t_array[i]
    d1, _ = calculate_d1_d2(path1[i], K, tau, r, sigma)

    delta.append(norm.cdf(d1))

option_premium, _ = black_scholes_price(S0, K, T, r, sigma)

cash = option_premium - (S0 * delta[0])
cash *= np.exp(r * delta_t)

for i in range(1, len(delta)):

    change = delta[i] - delta[i-1]

    cash -= path1[i] * change

    cash *= np.exp(r * delta_t)

value = cash + path1[-1] * delta[-1]

call_payoff = max(path1[-1] - K, 0)

error = value - call_payoff
print(error)

In [92]:
prices2 = prices[:, :-1]

tau = 1 - t_array[:-1]
d1, _ = calculate_d1_d2(prices2, K, tau, r, sigma)
delta = norm.cdf(d1)

option_premium, _ = black_scholes_price(S0, K, T, r, sigma)

cash = option_premium - (S0 * delta.T[0])
cash *= np.exp(r * delta_t)

for i in range(1, len(delta.T)):

    change = delta.T[i] - delta.T[i-1]

    cash -= prices2.T[i] * change

    cash *= np.exp(r * delta_t)

value = cash + (prices.T[-1] * delta.T[-1])

call_payoff = np.maximum(prices.T[-1] - K, 0)

error = value - call_payoff
print(error)

[-0.56821752  0.06377512 -0.03627471 -0.17636565 -0.22449833 -0.11875045
 -0.16854641 -0.64346544  0.79472094 -0.52599123]


In [ ]:
plt.hist(error,
        bins = 75
         )

plt.xlabel("Errors")
plt.ylabel("Frequency")
plt.title("Distribution of Hedging Errors")

plt.show()

In [27]:
tau = 1 - t_array[:-1]
d1, _ = calculate_d1_d2(prices[:, :-1], K, tau, r, sigma)
delta = norm.cdf(d1)

option_premium, _ = black_scholes_price(S0, K, T, r, sigma)

delta_change = np.diff(delta, axis = 1)
interest = np.exp(r * tau[1:])

rebalance_costs = delta_change * prices[:, 1:-1]
adjusted_costs = rebalance_costs * interest
total_cost = np.sum(adjusted_costs, axis=1)

initial_cash = option_premium - (prices.T[0] * delta.T[0])
initial_cash *= np.exp(r * T)

terminal_cash = initial_cash - total_cost

portfolio_value = terminal_cash + prices[:, -1] * delta[:, -1]

call_payoff = np.maximum(prices.T[-1] - K, 0)

error = portfolio_value - call_payoff


In [ ]:
# Slow nested loop version

option_premium, _ = black_scholes_price(S0, K, T, r, sigma)

errors_loop = []

for j in range(M):

    S = S0

    d1, _ = calculate_d1_d2(S, K, T, r, sigma)
    delta_old = norm.cdf(d1)

    cash = option_premium - S0 * delta_old

    for i in range(1, n + 1):

        S *= np.exp(
            (mu - 0.5 * sigma**2) * delta_t
            + sigma * randoms[j, i-1]
        )

        cash *= np.exp(r * delta_t)

        if i < n:
            tau = T - i * delta_t

            d1, _ = calculate_d1_d2(S, K, tau, r, sigma)
            delta_new = norm.cdf(d1)

            cash -= S * (delta_new - delta_old)
            delta_old = delta_new

    value = cash + S * delta_old
    payoff = max(S - K, 0)

    errors_loop.append(value - payoff)